# 15 - V2 data preparation

Prepare versioned leakage-safe splits. The original four datasets are corrective development evidence; CICIoT2023 and TON_IoT are reserved external confirmations.


In [4]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').is_dir():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Canonical RareGuard-XIDS IMPLEMENTATION project not found.')
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [7]:
!python -m rareguard.experiments.prepare_publication_data --config configs/publication_v2.yaml --datasets configs/datasets_v2.yaml --dataset all --resume


2026-10-23 15:49:40,271 | INFO | rareguard | Using mounted Google Drive dataset root /content/drive/MyDrive/projects/Dataset/Security
2026-10-23 15:49:40,274 | INFO | rareguard | Skipping completed publication split cse_cic_ids2018
2026-10-23 15:49:40,275 | INFO | rareguard | Skipping completed publication split cic_bccc_nrc_tabulariotattack_2024
2026-10-23 15:49:40,277 | INFO | rareguard | Skipping completed publication split cic_iiot_2025_datasense
2026-10-23 15:49:40,278 | INFO | rareguard | Skipping completed publication split edge_iiotset
Sampled CICIoT2023/csv/CICIoT2023/part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv: observed=28,452 retained=2,000 rows
Sampled CICIoT2023/csv/CICIoT2023/part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv: observed=28,516 retained=2,000 rows
Sampled CICIoT2023/csv/CICIoT2023/part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv: observed=28,476 retained=2,000 rows
Sampled CICIoT2023/csv/CICIoT2023/part-00003-363d1ba3-8ab5-4f96-bc

In [10]:
from pathlib import Path
import json
for path in sorted(Path('artifacts/publication_v2/processed').glob('*/split_metadata.json')):
    data = json.loads(path.read_text())
    print(path.parent.name, data['split_rows'], 'features=', data['features'], 'classes=', data['classes'], 'ready=', data['ready'])


cic_bccc_nrc_tabulariotattack_2024 {'train': 24829, 'validation': 3548, 'test': 7095} features= 79 classes= 15 ready= True
cic_iiot_2025_datasense {'train': 9099, 'validation': 1300, 'test': 2600} features= 71 classes= 39 ready= True
ciciot2023 {'train': 42000, 'validation': 6000, 'test': 12000} features= 46 classes= 34 ready= True
cse_cic_ids2018 {'train': 14794, 'validation': 2114, 'test': 4228} features= 78 classes= 13 ready= True
edge_iiotset {'train': 12187, 'validation': 1742, 'test': 3483} features= 53 classes= 11 ready= True
ton_iot {'train': 1399, 'validation': 200, 'test': 400} features= 15 classes= 10 ready= True


In [9]:
# Rebuild TON_IoT with full-file coverage after validating the four-window sample was under-supported.
!python -m rareguard.experiments.prepare_publication_data --config configs/publication_v2.yaml --datasets configs/datasets_v2.yaml --dataset ton_iot --force

2026-10-23 15:59:48,889 | INFO | rareguard | Using mounted Google Drive dataset root /content/drive/MyDrive/projects/Dataset/Security
Sampled TON_IoT/extracted/TON_IoT_Train_Test_Network.csv: observed=461,043 retained=2,000 rows
2026-10-23 15:59:56,663 | INFO | rareguard | ton_iot rows=1999 features=15 classes=10 ready=True
